# 09 — Remaining Trace Prediction (Seq2Seq)

Encoder-decoder architecture: given a prefix sequence, generate the remaining activity sequence token by token.  
Two models are implemented:
1. **LSTM Seq2Seq** — LSTM encoder + LSTM decoder with teacher forcing during training, greedy decoding at inference.
2. **Transformer + LSTM Decoder** — Transformer encoder + LSTM decoder.

Special tokens: `<PAD>=0`, `<UNK>=1`, `<SOS>=2`, `<EOS>=3`.  
Metrics: Damerau-Levenshtein Similarity (DLS), exact sequence match, next-activity accuracy.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, random
import pm4py as pm
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math

## 2. Load & Clean

In [ ]:
dtype_dict = {
    'hired': 'Int8', 'Rejected': 'Int8', 'CW': 'Int8', 'Evergreen': 'Int8',
    'Region': 'Int16', 'Country': 'Int16',
    'Job Family': 'Int16', 'Job Family Group': 'Int16',
}
df = pd.read_csv('data/'+
    'event_log_consolidated.csv',
    low_memory=False,
    dtype=dtype_dict,
    parse_dates=['timestamp']
)
df = df[~df['Step'].str.match(r'^\d+$', na=False)].copy()
df.sort_values(['Case_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape: {df.shape}')
print(f'Cases: {df["Case_id"].nunique():,}')

## 3. Format for pm4py + Chronological 60/20/20 Split

In [ ]:
df = pm.format_dataframe(
    df, case_id='Case_id', activity_key='Step', timestamp_key='timestamp'
)

case_start = df.groupby('case:concept:name')['time:timestamp'].min().sort_values()
n_cases      = len(case_start)
train_cutoff = case_start.iloc[int(n_cases * 0.60)]
val_cutoff   = case_start.iloc[int(n_cases * 0.80)]

train_cases = case_start[case_start <  train_cutoff].index
val_cases   = case_start[(case_start >= train_cutoff) & (case_start < val_cutoff)].index
test_cases  = case_start[case_start >= val_cutoff].index

print(f'Train: {len(train_cases):,} | Val: {len(val_cases):,} | Test: {len(test_cases):,}')
print(f'Train cutoff: {train_cutoff.date()} | Val cutoff: {val_cutoff.date()}')

train_df = df[df['case:concept:name'].isin(train_cases)].copy()
val_df   = df[df['case:concept:name'].isin(val_cases)].copy()
test_df  = df[df['case:concept:name'].isin(test_cases)].copy()

## 4. Remove Leaky Activities + Build Vocabulary

Special tokens: `<PAD>=0`, `<UNK>=1`, `<SOS>=2`, `<EOS>=3`. Activities start from index 4.

In [ ]:
LEAKY_THRESHOLD = 0.85
act_hire_rate = (
    train_df.groupby('concept:name')['hired']
    .apply(lambda x: x.astype(float).mean())
    .sort_values(ascending=False)
)
leaky_acts = set(act_hire_rate[act_hire_rate >= LEAKY_THRESHOLD].index)
print(f'Leaky activities removed ({len(leaky_acts)}):')
for act in sorted(leaky_acts, key=lambda a: -act_hire_rate[a]):
    print(f'  {act_hire_rate[act]:.3f}  {act}')

train_df = train_df[~train_df['concept:name'].isin(leaky_acts)].copy()
val_df   = val_df[~val_df['concept:name'].isin(leaky_acts)].copy()
test_df  = test_df[~test_df['concept:name'].isin(leaky_acts)].copy()

# Special tokens
PAD_IDX = 0
UNK_IDX = 1
SOS_IDX = 2
EOS_IDX = 3

all_train_acts = sorted(train_df['concept:name'].dropna().unique())
act2idx = {act: i+4 for i, act in enumerate(all_train_acts)}
act2idx['<PAD>'] = PAD_IDX
act2idx['<UNK>'] = UNK_IDX
act2idx['<SOS>'] = SOS_IDX
act2idx['<EOS>'] = EOS_IDX
idx2act = {v: k for k, v in act2idx.items()}
VOCAB_SIZE  = len(act2idx)
MAX_SEQ_LEN = 30
print(f'\nVocabulary size: {VOCAB_SIZE} | MAX_SEQ_LEN: {MAX_SEQ_LEN}')
print(f'PAD={PAD_IDX}  UNK={UNK_IDX}  SOS={SOS_IDX}  EOS={EOS_IDX}')

## 5. Dataset Construction

For prefix length k:
- **Encoder input**: activities at positions [0 … k-1] (the prefix), padded to MAX_SEQ_LEN
- **Decoder input**: `[SOS] + suffix activities` (teacher forcing target shifted right)
- **Decoder target**: `suffix activities + [EOS]`

In [ ]:
PREFIX_LENGTHS = [1, 3, 5, 10, 20]


def build_s2s_samples(df_subset, k, act2idx, max_seq_len):
    """
    Build encoder inputs, decoder inputs, and decoder targets for prefix length k.
    Returns arrays: (enc_inputs, dec_inputs, dec_targets, case_ids)
    """
    enc_inputs, dec_inputs, dec_targets, cids = [], [], [], []
    df_s = df_subset.sort_values(['case:concept:name', 'time:timestamp'])

    for cid, grp in df_s.groupby('case:concept:name', sort=False):
        grp = grp.sort_values('time:timestamp')
        acts = grp['concept:name'].tolist()
        if len(acts) <= k:
            continue  # no suffix

        prefix  = acts[:k]
        suffix  = acts[k:]

        # Encoder: prefix padded left
        enc_idxs = [act2idx.get(a, UNK_IDX) for a in prefix]
        enc_idxs = enc_idxs[-max_seq_len:]  # truncate
        enc_pad  = [PAD_IDX] * (max_seq_len - len(enc_idxs)) + enc_idxs

        # Decoder input:  SOS + suffix (truncated)
        suf_idxs = [act2idx.get(a, UNK_IDX) for a in suffix]
        suf_idxs = suf_idxs[:max_seq_len]  # keep start
        dec_in   = [SOS_IDX] + suf_idxs
        dec_tgt  = suf_idxs + [EOS_IDX]

        # Pad decoder sequences to max_seq_len + 1
        dec_len  = max_seq_len + 1
        dec_in  += [PAD_IDX] * (dec_len - len(dec_in))
        dec_tgt += [PAD_IDX] * (dec_len - len(dec_tgt))

        enc_inputs.append(enc_pad)
        dec_inputs.append(dec_in)
        dec_targets.append(dec_tgt)
        cids.append(cid)

    return (
        np.array(enc_inputs,  dtype=np.int64),
        np.array(dec_inputs,  dtype=np.int64),
        np.array(dec_targets, dtype=np.int64),
        cids
    )


class S2SDataset(Dataset):
    def __init__(self, enc, dec_in, dec_tgt):
        self.enc     = torch.tensor(enc,     dtype=torch.long)
        self.dec_in  = torch.tensor(dec_in,  dtype=torch.long)
        self.dec_tgt = torch.tensor(dec_tgt, dtype=torch.long)
    def __len__(self): return len(self.enc)
    def __getitem__(self, i): return self.enc[i], self.dec_in[i], self.dec_tgt[i]


print('S2S dataset builder defined.')
print(f'Will train on prefix lengths: {PREFIX_LENGTHS}')

## 6. Evaluation Metrics

In [ ]:
def damerau_levenshtein(a, b):
    """DL distance on integer lists (activity index sequences)."""
    n, m = len(a), len(b)
    d = np.zeros((n+1, m+1), dtype=int)
    for i in range(n+1): d[i, 0] = i
    for j in range(m+1): d[0, j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            d[i, j] = min(d[i-1, j]+1, d[i, j-1]+1, d[i-1, j-1]+cost)
            if i > 1 and j > 1 and a[i-1] == b[j-2] and a[i-2] == b[j-1]:
                d[i, j] = min(d[i, j], d[i-2, j-2]+cost)
    return d[n, m]


def trim_sequence(seq):
    """Remove PAD and EOS tokens; keep until first EOS."""
    result = []
    for t in seq:
        if t == EOS_IDX:
            break
        if t not in (PAD_IDX, SOS_IDX):
            result.append(t)
    return result


def evaluate_s2s(pred_seqs, target_seqs, sample=1000):
    """Compute DLS, exact match, and next-activity accuracy."""
    assert len(pred_seqs) == len(target_seqs)
    idx = np.random.choice(len(pred_seqs), size=min(sample, len(pred_seqs)), replace=False)
    dls_vals, exact, next_act_correct = [], 0, 0
    for i in idx:
        p = trim_sequence(pred_seqs[i])
        t = trim_sequence(target_seqs[i])
        dist = damerau_levenshtein(t, p)
        max_len = max(len(t), len(p), 1)
        dls_vals.append(1.0 - dist / max_len)
        exact += int(t == p)
        next_act_correct += int(len(t) > 0 and len(p) > 0 and t[0] == p[0])
    n = len(idx)
    return {
        'DLS':      float(np.mean(dls_vals)),
        'Exact':    exact / n,
        'NextAcc':  next_act_correct / n,
    }


print('Metrics defined.')

## 7. LSTM Seq2Seq Model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(torch.__version__)


class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=256, n_layers=2, dropout=0.3):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)

    def forward(self, src):
        embedded = self.emb(src)
        outputs, (h, c) = self.lstm(embedded)
        return outputs, h, c


class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=256, n_layers=2, dropout=0.3):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc   = nn.Linear(hidden, vocab_size)

    def forward(self, tgt_token, h, c):
        # tgt_token: (B,) -> (B, 1)
        embedded = self.emb(tgt_token.unsqueeze(1))
        out, (h, c) = self.lstm(embedded, (h, c))
        logits = self.fc(out.squeeze(1))  # (B, vocab_size)
        return logits, h, c


class LSTMSeq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=256, n_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = LSTMEncoder(vocab_size, emb_dim, hidden, n_layers, dropout)
        self.decoder = LSTMDecoder(vocab_size, emb_dim, hidden, n_layers, dropout)

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """
        src: (B, src_len)
        tgt: (B, tgt_len)  — includes SOS at position 0
        Returns logits: (B, tgt_len-1, vocab_size)
        """
        _, h, c = self.encoder(src)
        tgt_len = tgt.size(1)
        all_logits = []
        dec_input = tgt[:, 0]  # SOS
        for t in range(1, tgt_len):
            logits, h, c = self.decoder(dec_input, h, c)
            all_logits.append(logits.unsqueeze(1))
            teacher_force = random.random() < teacher_forcing_ratio
            dec_input = tgt[:, t] if teacher_force else logits.argmax(1)
        return torch.cat(all_logits, dim=1)  # (B, tgt_len-1, vocab)

    def greedy_decode(self, src, max_len=MAX_SEQ_LEN):
        """Greedy inference — no teacher forcing."""
        _, h, c = self.encoder(src)
        dec_input = torch.full((src.size(0),), SOS_IDX, dtype=torch.long, device=device)
        preds = []
        for _ in range(max_len):
            logits, h, c = self.decoder(dec_input, h, c)
            dec_input = logits.argmax(1)
            preds.append(dec_input.unsqueeze(1))
        return torch.cat(preds, dim=1)  # (B, max_len)


print('LSTM Seq2Seq defined.')

## 8. Transformer Encoder + LSTM Decoder Model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerLSTMSeq2Seq(nn.Module):
    """
    Transformer encoder (self-attention on prefix) +
    LSTM decoder (autoregressive suffix generation).
    """
    def __init__(self, vocab_size, emb_dim=64, nhead=4, n_enc_layers=2,
                 n_dec_layers=2, d_ff=128, hidden=256, dropout=0.1):
        super().__init__()
        # Encoder
        self.src_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.pe      = PositionalEncoding(emb_dim, dropout=dropout)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=nhead, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_enc_layers)
        self.enc2dec = nn.Linear(emb_dim, hidden)  # bridge to LSTM hidden

        # Decoder (LSTM)
        self.tgt_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm    = nn.LSTM(emb_dim, hidden, num_layers=n_dec_layers,
                               batch_first=True,
                               dropout=dropout if n_dec_layers > 1 else 0)
        self.fc      = nn.Linear(hidden, vocab_size)
        self.n_dec_layers = n_dec_layers
        self.hidden  = hidden

    def _encode(self, src):
        pad_mask = (src == PAD_IDX)
        mem = self.encoder(self.pe(self.src_emb(src)), src_key_padding_mask=pad_mask)
        # Use mean pooling of encoder output as initial decoder hidden
        mask_f = (~pad_mask).unsqueeze(-1).float()
        ctx = (mem * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)  # (B, emb_dim)
        h0  = torch.tanh(self.enc2dec(ctx))                         # (B, hidden)
        h0  = h0.unsqueeze(0).expand(self.n_dec_layers, -1, -1).contiguous()
        c0  = torch.zeros_like(h0)
        return h0, c0

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        h, c = self._encode(src)
        tgt_len = tgt.size(1)
        all_logits = []
        dec_input = tgt[:, 0]
        for t in range(1, tgt_len):
            emb = self.tgt_emb(dec_input.unsqueeze(1))
            out, (h, c) = self.lstm(emb, (h, c))
            logits = self.fc(out.squeeze(1))
            all_logits.append(logits.unsqueeze(1))
            teacher_force = random.random() < teacher_forcing_ratio
            dec_input = tgt[:, t] if teacher_force else logits.argmax(1)
        return torch.cat(all_logits, dim=1)

    def greedy_decode(self, src, max_len=MAX_SEQ_LEN):
        h, c = self._encode(src)
        dec_input = torch.full((src.size(0),), SOS_IDX, dtype=torch.long, device=device)
        preds = []
        for _ in range(max_len):
            emb = self.tgt_emb(dec_input.unsqueeze(1))
            out, (h, c) = self.lstm(emb, (h, c))
            dec_input = self.fc(out.squeeze(1)).argmax(1)
            preds.append(dec_input.unsqueeze(1))
        return torch.cat(preds, dim=1)


print('Transformer+LSTM Seq2Seq defined.')

## 9. Training Loop

Train each model for each prefix length k. Loss: cross-entropy ignoring PAD tokens.

In [ ]:
def train_s2s_model(model, train_loader, val_loader, n_epochs=20):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    best_loss, best_state, no_improve = float('inf'), None, 0

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0.0
        for enc_b, dec_in_b, dec_tgt_b in train_loader:
            enc_b, dec_in_b, dec_tgt_b = enc_b.to(device), dec_in_b.to(device), dec_tgt_b.to(device)
            optimizer.zero_grad()
            logits = model(enc_b, dec_in_b, teacher_forcing_ratio=0.5)
            # logits: (B, tgt_len-1, vocab_size)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), dec_tgt_b[:, :T].reshape(B*T))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for enc_b, dec_in_b, dec_tgt_b in val_loader:
                enc_b, dec_in_b, dec_tgt_b = enc_b.to(device), dec_in_b.to(device), dec_tgt_b.to(device)
                logits = model(enc_b, dec_in_b, teacher_forcing_ratio=0.0)
                B, T, V = logits.shape
                val_loss += criterion(logits.reshape(B*T, V), dec_tgt_b[:, :T].reshape(B*T)).item()
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        print(f'  Epoch {epoch+1:02d}  val_loss={val_loss:.4f}')

        if val_loss < best_loss:
            best_loss  = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 5:
                print(f'  Early stop at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model


def run_inference(model, loader):
    """Greedy decode all test samples; return (pred_seqs, tgt_seqs) as lists of lists."""
    model.eval()
    all_preds, all_tgts = [], []
    with torch.no_grad():
        for enc_b, dec_in_b, dec_tgt_b in loader:
            enc_b = enc_b.to(device)
            preds = model.greedy_decode(enc_b, max_len=MAX_SEQ_LEN)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_tgts.extend(dec_tgt_b.numpy().tolist())
    return all_preds, all_tgts


print('Training loop defined.')

## 10. Train and Evaluate per Prefix Length

In [ ]:
results = {}  # (model_name, k) -> metrics dict

for k in PREFIX_LENGTHS:
    print(f'\n══════════════════════════════')
    print(f'  Prefix length k = {k}')
    print(f'══════════════════════════════')

    enc_tr, dec_in_tr, dec_tgt_tr, _ = build_s2s_samples(train_df, k, act2idx, MAX_SEQ_LEN)
    enc_va, dec_in_va, dec_tgt_va, _ = build_s2s_samples(val_df,   k, act2idx, MAX_SEQ_LEN)
    enc_te, dec_in_te, dec_tgt_te, _ = build_s2s_samples(test_df,  k, act2idx, MAX_SEQ_LEN)

    if len(enc_tr) == 0 or len(enc_te) == 0:
        print('  Skipping (empty split)')
        continue

    tr_ds  = S2SDataset(enc_tr, dec_in_tr, dec_tgt_tr)
    va_ds  = S2SDataset(enc_va, dec_in_va, dec_tgt_va)
    te_ds  = S2SDataset(enc_te, dec_in_te, dec_tgt_te)
    tr_ldr = DataLoader(tr_ds, batch_size=256, shuffle=True,  num_workers=0)
    va_ldr = DataLoader(va_ds, batch_size=512, shuffle=False, num_workers=0)
    te_ldr = DataLoader(te_ds, batch_size=512, shuffle=False, num_workers=0)

    print(f'  Samples — Train: {len(tr_ds):,}  Val: {len(va_ds):,}  Test: {len(te_ds):,}')

    for model_name, ModelClass, kwargs in [
        ('LSTM-S2S',    LSTMSeq2Seq,          dict(vocab_size=VOCAB_SIZE)),
        ('TF-LSTM-S2S', TransformerLSTMSeq2Seq, dict(vocab_size=VOCAB_SIZE)),
    ]:
        print(f'\n  [{model_name}] Training...')
        model = ModelClass(**kwargs).to(device)
        model = train_s2s_model(model, tr_ldr, va_ldr, n_epochs=20)

        preds, tgts = run_inference(model, te_ldr)
        metrics = evaluate_s2s(preds, tgts)
        results[(model_name, k)] = metrics
        print(f'  [{model_name}] k={k}  DLS={metrics["DLS"]:.4f}  '
              f'Exact={metrics["Exact"]:.4f}  NextAcc={metrics["NextAcc"]:.4f}')

## 11. Results Table + DLS vs Prefix Plot

In [ ]:
print('\n=== SEQ2SEQ RESULTS ===')
for (mname, k), m in sorted(results.items(), key=lambda x: (x[0][0], x[0][1])):
    print(f'{mname:15s}  k={str(k):3s}  DLS={m["DLS"]:.4f}  '
          f'Exact={m["Exact"]:.4f}  NextAcc={m["NextAcc"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_names = ['LSTM-S2S', 'TF-LSTM-S2S']
metric_keys = [('DLS', 'DLS Similarity'), ('Exact', 'Exact Match'), ('NextAcc', 'Next-Act Accuracy')]
for ax, (mk, title) in zip(axes, metric_keys):
    for mname in model_names:
        vals = [results.get((mname, k), {}).get(mk, np.nan) for k in PREFIX_LENGTHS]
        ax.plot([str(k) for k in PREFIX_LENGTHS], vals, marker='o', label=mname)
    ax.set_title(f'Remaining Trace S2S: {title}')
    ax.set_xlabel('Prefix Length'); ax.set_ylabel(title); ax.legend()
plt.tight_layout()
plt.savefig('figs/remaining_trace_s2s_metrics.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved figs/remaining_trace_s2s_metrics.png')